# MicaSense Image Processing Setup

## Overview

This series of tutorials will be a walk through on how to process RedEdge data from raw images through conversion to reflectance. In this first tutorial, we will cover the tools required to do this, get them installed, and verify that the installation works.

## System Requirements

Our tutorials are written using Python 3.10+. Python has great library support for image processing through libraries such as OpenCV, SciKit Image, and others. In this tutorial, we'll use python, OpenCV, numpy, and matplotlib, as well as the standalone exiftool and its python wrapper to open and manipulate RedEdge images to transform raw digital number values into quantitative reflectance.

This tutorial has been tested on Windows, MacOS, and Linux. It is likely to work on other platforms, especially unix-based platforms like OSX, but you will have to do the legwork to get the required software installed and working.

### Software/Libraries Overview

The following software and libraries are required for this tutorial:

* [python3](https://www.python.org/downloads/)
* [numpy](https://www.numpy.org/)
* [openCV](https://opencv.org/)
* [matplotlib](https://matplotlib.org/)
* [exiftool](https://exiftool.org/) + [pyexiftool](https://github.com/smarnach/pyexiftool)
* [scikit-image](https://scikit-image.org/)
* [zbar](https://zbar.sourceforge.net/) + [pyzbar](https://github.com/NaturalHistoryMuseum/pyzbar)
* [pysolar](http://pysolar.org/)
* [GDAL](https://gdal.org/) (system package; match the pip `gdal` wheel to your installed version)
* [pandas](https://pandas.pydata.org/) (notebooks)
* [jupyter](https://jupyter.org/) (notebooks)

Install the library and notebook extras from a clone of this repository:

    git clone https://github.com/brainergylab/micasense_imageprocessing.git
    cd micasense_imageprocessing
    python3 -m venv .venv
    source .venv/bin/activate   # Windows: .venv\Scripts\activate
    pip install -e ".[test]"
    pip install jupyter pandas imageio

On Linux, install system packages before `pip install` (see CI in `.github/workflows/python-test.yml`):

    sudo apt-get update
    sudo apt-get install exiftool libgdal-dev libzbar0
    GDAL_VERSION=$(gdal-config --version)
    pip install "gdal==$GDAL_VERSION"

### Linux (Debian/Ubuntu)

The following is what we use on a fresh Ubuntu image. First install system tools and libraries:

    sudo apt install git git-lfs libzbar0 libgdal-dev exiftool make

Clone the repository, create a virtual environment, and install Python dependencies as shown above.

Verify the install:

    pytest

Or start the notebooks:

    jupyter notebook .

### Windows setup

* Install [Python 3](https://www.python.org/downloads/) for your system.
* Download the [exiftool windows package](https://exiftool.org/) and unzip it to a permanent location such as `c:\exiftool\`.
* Add an environment variable `exiftoolpath` pointing to `c:\exiftool\exiftool.exe` (Settings → Environment Variables → New).
* Open Command Prompt or PowerShell, `cd` to the repository, create and activate a venv, then run `pip install -e ".[test]"` and install GDAL matching your environment if needed.

### MacOS setup

* Install [git](https://git-scm.com/downloads) and [git-lfs](https://git-lfs.github.com/).
* Install [exiftool](https://exiftool.org/).
* Install [Homebrew](https://brew.sh/) and `zbar` / GDAL as needed: `brew install zbar gdal`
* Create a virtual environment in the repository and `pip install -e ".[test]"` as above.

## Running the notebooks

* Activate your virtual environment (`source .venv/bin/activate` or `.venv\Scripts\activate` on Windows)
* `cd` to the repository directory
* Run `jupyter notebook .`

## Testing Installation

The following python snippet can be run from a jupyter notebook, inside iPython, or by saving to a script and running from the command line. If you're on windows, make sure you have set the location of exiftool in the `exiftoolpath` environment variable. If this script succeeds, your system is ready to go! If not, check the installation documentation for the module import that is having issues.



In [ ]:
import exiftool
import os
import glob

print()
print("Successfully imported all required libraries.")
print()

if os.name == "nt":
    if os.environ.get("exiftoolpath") is None:
        print("Set the `exiftoolpath` environment variable as described above")
    else:
        if not os.path.isfile(os.environ.get("exiftoolpath")):
            print("The provided exiftoolpath isn't a file, check the settings")

try:
    with exiftool.ExifTool(os.environ.get("exiftoolpath")) as exift:
        print("Successfully executed exiftool.")
except Exception as e:
    print(
        "Exiftool isn't working. Double check that you've followed the instructions above."
    )
    print("The execption text below may help to find the source of the problem:")
    print()
    print(e)

### Testing image reading and panel detection

The above code checks for the proper libraries to be installed and verifies it can execute `exiftool`. This code opens an example image, reads the metadata, and then uses the `pyzbar` library to find a MicaSense panel in the image. 

In [ ]:
from micasense.image import Image

imagePath = os.path.join(".", "data", "REDEDGE-MX")
imageName = glob.glob(os.path.join(imagePath, "IMG_0001_1.tif"))[0]

img = Image(imageName)
img.plot_raw(figsize=(8.73, 8.73))

from micasense.panel import Panel

panel = Panel(img)
if not panel.panel_detected():
    raise IOError("Panel Not Detected! Check your installation of pyzbar")
else:
    panel.plot(figsize=(8, 8))

print("Success! Now you are ready for Part 1 of the tutorial.")

---
Copyright (c) 2017-2019 MicaSense, Inc.  For licensing information see the [project git repository](https://github.com/micasense/imageprocessing)